In [ ]:
import pickle
import seaborn
import matplotlib
import matplotlib.pyplot as plt
import os
import mne
import numpy as np
import pandas as pd
import torch
import copy

from omegaconf import OmegaConf
import yaml
import argparse
from data.data_loader import load_eeg_data

General Goal of this Notebook:\
Check if removing outliers in variance (top 5%) has effect on the topoplots when we group by uncertainty.\
This is done because in uncertainty_stat_analysis.ipynb it was observed that while the majority of variances in trials are <=1, there are some extremely high outliers for some trials, especially for some of the subjects.\
Also I want to check whether aggregation over the time_point and trial dimension with mean leads to vastly different result that only aggregation over the time_point dimension with mean and over the trial dimension linearly, via a top-k approach.\
The idea is that with a top-k approach, every trial is equally important, reducing the influence of potential outlier trials

# Boilerplate code

In [ ]:
CFG_YAML = """
wandb:
 key: f0c92a0059bf12e2647f0a1c22fdcd12555fa6df
model:
dataset:
 data_directory: /home/marco/Documents/GitHub/tms_eeg_decoding/data
 #file_name: file_name: subject_{:03d}_preprocessed_combined_py.fif
 file_name: subject_{:03d}_preprocessed_combined_py.fif
 exclude_timepoints: 100
 subject_index: 1
 test_subject_indices: [2,4,7,9,11,13,14,18,22,24]
 #test_subject_indices: [2,4,7,9,11,13,14,18,22,24]
training:
 training_start_len: 100
 pretrain_epochs: 100
 pretrain_lr: 0.0001
 val_window_len: 1
 epochs_per_window: 10
 num_warmup_epochs: 5
 num_epochs: 800
 slide_step: 1
 num_warmup_epochs_per_window: 0
 lr: 0.005 #maybe change back to 0.0001
 nll_beta: 0.001
 num_warmup_epochs: 0
 batch_size: 50 #better to use 50
 random_seed: 42
 precision: bf16
 kde_lambda: 0.5
 finetune_entire_model: true # Set to true to finetune the entire model, false for transformer only
exp_name: S4_S4EEGNet_ema
"""


In [ ]:
def load_config():
    cfg = OmegaConf.create(yaml.safe_load(CFG_YAML))
    cfg.exp_name = f"{cfg.exp_name}_subject_{cfg.dataset.subject_index}"
    return cfg

def parse_args():
    parser = argparse.ArgumentParser()
    parser.add_argument("--update_conf", nargs="*", help="Updates to the configuration in the form of key=value pairs", default=[])
    parser.add_argument("-f", "--fff", help="A dummy argument to handle IPython's default argument", default="1")
    return parser.parse_args()

def update_config(cfg, cli_args):
    for update in cli_args.update_conf:
        key, value = update.split("=")
        try:
            value = eval(value)
        except:
            pass
        OmegaConf.update(cfg, key, value, force_add=True)
    cfg.exp_name = cfg.exp_name + "_" + "_".join(cli_args.update_conf)
    print(OmegaConf.to_yaml(cfg))
    return cfg


def save_config(cfg):
    os.makedirs("conf/sweeps", exist_ok=True)
    os.makedirs("exp/withinsubs", exist_ok=True)
    with open(f"conf/sweeps/withinsubs_{cfg.exp_name}.yaml", "w") as f:
        f.write(OmegaConf.to_yaml(cfg))


In [ ]:

# award importance to a channel according to their importance ranking in a time point
def update_channel_points_linear(top_k_channel_names, channel_point_dict, k):
    for i, channel_name in enumerate(top_k_channel_names):
        channel_point_dict[channel_name] += k-i
    return channel_point_dict

# award importance to a channel according to their importance ranking in a time point
def update_channel_points_same(top_k_channel_names, channel_point_dict, k):
    for i, channel_name in enumerate(top_k_channel_names):
        channel_point_dict[channel_name] += 1
    return channel_point_dict

def get_top_k_channels_per_timepoint(time_point_data, ch_names, k):
    # trial is of shape channels x timepoins
    # for each timepoint, find indices of top_k channels with highest activations 
    top_k_channels_indices = np.argsort(time_point_data)[::-1][:k]
    # get the channel names of the top_k channels 
    top_k_channels_names = np.take(ch_names, top_k_channels_indices)
    #print(top_k_channels_names)
    return top_k_channels_names


# adapt so you can see a running list of top channes per trial
def get_top_k_channels_per_trial(trial, ch_names, channel_point_dict , k, update_func):
    for time_point in range(trial.shape[1]):
        #print(f"time_point: {time_point}")
        top_k_channel_names = get_top_k_channels_per_timepoint(trial[:,time_point], ch_names, k)
        # update the points of the top_k channels of the timepoint according to the (linear, stricly monotone falling) update function
        # 
        # NOTE while assumption that all trials are equally important holds, assumption that all timepoints are equally important is very questionable. 
        # Maybe think about this part again
        channel_point_dict = update_func(top_k_channel_names, channel_point_dict)
    return channel_point_dict

def get_top_k_channels(aggregate_dict, aggregate_dict_key, ch_names, k, update_func):
    # get the channel importance for each channel according to the explanation method
    # a ranking approach is chosen s.t. each timepoint in a trial and each trial is equally important

    # init dict for storing points per channel

    channel_point_dicts_over_time = []

    channel_point_dict = {}
    for ch in ch_names:
        channel_point_dict[ch] = 0
    
    data = aggregate_dict[aggregate_dict_key].squeeze()
    # go through each trial in the dataset and assign points for each
    for trial in data:
        #print(trial.shape)
        channel_point_dict = get_top_k_channels_per_trial(trial, ch_names, channel_point_dict, k, update_func)
        channel_point_dicts_over_time.append(copy.deepcopy(channel_point_dict))
        #print(channel_point_dicts_over_time)
    
    top_k_channels = sorted(channel_point_dict, key=channel_point_dict.get, reverse=True)[:k]
    return top_k_channels, channel_point_dicts_over_time
    
def format_point_progression(point_progressions, ch_names):
    # this function collects the points over time per channel properly s.t. the point progression over time can easily be plotted
    point_progression_dict = {}
    for ch in ch_names:
        point_progression_dict[ch] = []


    for dict in point_progressions:
        for ch in ch_names:
            point_progression_dict[ch].append(dict[ch])

    return point_progression_dict


def top_k_channel_per_trial_weighted(trial, ch_names,  channel_points_dict, k, update_func, aggregation_func=np.mean, **kwargs):
    # aggregate the importances award to each "pixel" in the trial over the time_point dimension according to the provided aggreagation_func
    #print(trial.shape)
    agg_points_per_channel = aggregation_func(trial, **kwargs)

    #print(agg_points_per_channel.shape)
    # get the indices of the top k channels
    top_k_channels_indices = np.argsort(agg_points_per_channel)[::-1][:k]
    top_k_channels_names = np.take(ch_names, top_k_channels_indices)
    #print(top_k_channels_names)
    # update the channels points according to the provided update func
    channel_points_dict = update_func(top_k_channels_names, channel_points_dict, k)
    return channel_points_dict

def get_top_k_weighted(aggregate_dict, aggregate_dict_key, ch_names, k, update_func, groupby_indices=np.array([])):
    #the previous top-k function considers each timepoint and each trial equally important.
    #in this version now each trial is still assumed to be equally important.
    #However, each timepoints importance is now weighted by the importance attributed to it by the attribution method.
    #we take the mean over the timepoint dimension. On one hand this increases susceptability to outliers, on the other hand it better allows
    #for some timepoints to be more important than others which I assume(for now) can absolutely be the case
    channel_point_dicts_over_time = []

    channel_point_dict = {}
    for ch in ch_names:
        channel_point_dict[ch] = 0

    if groupby_indices.any():
        data = aggregate_dict[aggregate_dict_key].squeeze()[groupby_indices]
    else:
        data = aggregate_dict[aggregate_dict_key].squeeze()
    # go through each trial in the dataset and assign points for each



    for trial in data:
        channel_point_dict = top_k_channel_per_trial_weighted(trial, ch_names, channel_point_dict, k, update_func, aggregation_func=np.mean, axis=1)
        channel_point_dicts_over_time.append(copy.deepcopy(channel_point_dict))

    top_k_channels = sorted(channel_point_dict, key=channel_point_dict.get, reverse=True)[:k]
    
    return top_k_channels, channel_point_dicts_over_time
    

def get_top_k_weighted_individual(aggregate_dict, aggregate_dict_key, ch_names, k, update_func, groupby_indices=np.array([])):
    # unlike the upper version which collects cumulative points per channel over trials
    # this function outputs the top-k chanels per trial for each trial individually
    channel_point_dicts_individual_trials = []

    if groupby_indices.any():
        data = aggregate_dict[aggregate_dict_key].squeeze()[groupby_indices]
    else:
        data = aggregate_dict[aggregate_dict_key].squeeze()

    for trial in data:
        channel_point_dict = {}
        for ch in ch_names:
            channel_point_dict[ch] = 0

        #print(trial.shape)
        channel_point_dict = top_k_channel_per_trial_weighted(trial, ch_names, channel_point_dict, k, update_func, axis=1)
        channel_point_dicts_individual_trials.append(copy.deepcopy(channel_point_dict))
        #print(channel_point_dicts_over_time)

    temp = format_point_progression(channel_point_dicts_individual_trials, ch_names)
    sum_over_channel_points_dict = {key: sum(value) for key,value in temp.items()}
    top_k_channels = sorted(sum_over_channel_points_dict, key=sum_over_channel_points_dict.get, reverse=True)[:k]

    return top_k_channels, channel_point_dicts_individual_trials


def top_k_groupby(aggregate_dict, aggregate_dict_key, groupby_dict, ch_names, k, update_func, top_k_func=get_top_k_weighted):
    groupby_channel_points_dict =  {}
    for groupby_key, groupby_indices in groupby_dict.items():
        groupby_channel_points_dict[groupby_key] = top_k_func(aggregate_dict, aggregate_dict_key, ch_names, k, update_func, groupby_indices)
    
    return groupby_channel_points_dict
 


In [ ]:
cwd = os.getcwd()

cwd = os.getcwd()
file_path = os.path.join(cwd, "../data/subject_007_preprocessed_combined_py.fif")
epochs = mne.read_epochs(file_path)
info = epochs.info
ch_names = epochs.ch_names

trial_numbers = []
cfg = load_config()
for subject_index in cfg.dataset.test_subject_indices:
    cfg = load_config()
    cfg.dataset.subject_index = subject_index
    cfg.exp_name = f"S4_S4EEGNet_ema_100_cal_py_{cfg.dataset.subject_index}"

    cli_args = parse_args()
    cfg = update_config(cfg, cli_args)
    save_config(cfg)

    all_epochs, all_labels_raw, fixed_median, fixed_q1, fixed_q3, labels_scaler, mean_mep, ch_names = load_eeg_data(cfg)
    trial_numbers.append(all_epochs.shape[0])

del all_epochs, all_labels_raw, fixed_median, fixed_q1, fixed_q3, labels_scaler, mean_mep,


subjects_dicts = []
subject_indices = [2,4,7,9,11,13,14,22,24]
for i in subject_indices:
    with open(f'./agg_dict_subject_{i}.pickle', 'rb') as handle:
        aggregate_dict = pickle.load(handle)
        subjects_dicts.append(aggregate_dict)


last_indices = []
for i in range(len(subjects_dicts)):
    end_idx = np.where(subjects_dicts[i]["dl_agg"]==0)[0][0]
    last_indices.append(end_idx)

for i in range(len(subjects_dicts)):
    for idx,(k,v) in enumerate(subjects_dicts[i].items()):
        subjects_dicts[i][k] = np.array(v[:last_indices[i]])

In [ ]:
def get_and_plot_topomap_groupby(aggregate_dict, groupby_dict, aggregate_dict_key, aggregation_func, info, subject_idx=2, aggregation_func_name="mean", groupby_name="uncertainty", save_path="refactor_test/groupby/uncertainty/topoplots/", **kwargs):
    # as the outputs of the XAI approaches come in the form trials x channels x timepoints, some form of summary stat needs to be applied.
    # this function allows for passing only a single trial and computing the mean over the timepoint dim,
    # or passing all trials and aggregating over trial and timepoints dim
    # Note: If only a single trial is passed, it needs to be passed in the shape 1 x channel x timepoints
    ncols = len(groupby_dict.keys())
    fig, axs = plt.subplots(nrows=1, ncols=ncols, figsize=(12,4))
    fig.suptitle(f"subject: {subject_idx}, aggregation: {aggregation_func_name}")
    for idx, groupby_key in enumerate(groupby_dict.keys()):
        #fig, ax = plt.subplots(figsize=(6,6))
        data = aggregate_dict[aggregate_dict_key].squeeze()
        #print(aggregation_func(data[groupby_dict[groupby_key]], **kwargs))
        axs[idx].set_title(f"{groupby_name}: {groupby_key}")
        mne.viz.plot_topomap(aggregation_func(data[groupby_dict[groupby_key]], **kwargs), info, axes=axs[idx], show=False)

        dir_path = f"{save_path}/{aggregation_func_name}"
        if not os.path.exists(dir_path):
            os.makedirs(dir_path)

    fig.savefig(f"{dir_path}/{aggregate_dict_key}_{aggregation_func_name}_{groupby_name}_subject_{subject_idx}_topoplot.png")

# Condtional Groups defintion

In [ ]:
groupby_dicts_list = []
for i in range(len(subject_indices)):
    remove_indices = []
    sigma2 = np.exp(subjects_dicts[i]["pred_uncertainty"])
    # throw away top 5% of values, hoping to remove outliers
    sigma2_indices = sigma2<=np.percentile(sigma2, 95)
    outlier_indices = ~sigma2_indices
    #hist_values.append(sigma2[i][vars_lower_x_percent_bool])   

    #This way the treshholds are calculated without the outliers
    low_threshold = np.percentile(sigma2[sigma2_indices], 33)
    high_threshold = np.percentile(sigma2[sigma2_indices], 66)
    # Treshholds are now defined without consideration of outliers.
    # BUT: outlier trials not removed from dataset!

    # Remove outliers by getting boolean array of them and using xor(^) with group_indices
    low_indices = (sigma2 < low_threshold)
    medium_indices = ((low_threshold<=sigma2) & (sigma2<=high_threshold))
    high_indices = (sigma2>high_threshold)^outlier_indices
    uncertainty_levels = {"low":low_indices, "medium":medium_indices, "high":high_indices}


    binary_label_fixed_zero_bool = subjects_dicts[i]["pred_binary_label_fixed"] == 0
    binary_label_fixed_one_bool = subjects_dicts[i]["pred_binary_label_fixed"] ==  1
    binary_label_fixed_dict = {"pred_zero" : binary_label_fixed_zero_bool, "pred_one" : binary_label_fixed_one_bool}

    binary_label_rolling_zero_bool = subjects_dicts[i]["pred_binary_label_rolling"] == 0
    binary_label_rolling_one_bool = subjects_dicts[i]["pred_binary_label_rolling"] == 1
    binary_label_rolling_dict = {"pred_zero" : binary_label_rolling_zero_bool , "pred_one" : binary_label_rolling_one_bool}

    binary_label_true_zero_bool = subjects_dicts[i]["true_binary_label"] ==  0 
    binary_label_true_one_bool = subjects_dicts[i]["true_binary_label"] ==  1 
    binary_label_true_dict = {"true_zero" : binary_label_true_zero_bool , "true_one" : binary_label_true_one_bool}
    
    groupby_dicts_list.append((uncertainty_levels,binary_label_fixed_dict, binary_label_rolling_dict, binary_label_true_dict))

    

# In gernal it seems like true_label zero seems to have a higher variation between channels while true label one is usually focused on one or two channels.


# groupby uncertainty

## aggregate over trial and timepoint dimension with mean

In [ ]:
for i in range(len(subject_indices)):
    get_and_plot_topomap_groupby(subjects_dicts[i], groupby_dicts_list[i][0], "dl_agg", np.mean, info, subject_idx=subject_indices[i], save_path="./groupby/Uncertainty/topoplots", axis=(0,2))

## aggregate over timepoint dimension with mean, and over trial dimension with top-k

### linear update function

The differenence between the two approaches (mean vs top-k) is that when we aggregate over the trial timension with top-k, we work the assumption into the aggregation that every trial is of equal importance to the result.

In every trial get the top k(=20) channels. Using the update function update_channel_points_linear we accumulate channel importances across trials. The top channel gets k points, the second channel k-1 points i.e. points awared are decreased linearly, Until a minimum of 0.

In [ ]:
points_progression_per_subject_uncertainty = []
for i in range(len(subject_indices)):
    points_progression_per_subject_uncertainty.append((top_k_groupby(subjects_dicts[i], "dl_agg", groupby_dicts_list[i][0], ch_names, 10, update_channel_points_linear)))

code up one version that just takes accumulated points index(for linear update funciont) and one that accumulates over trial dimension( for same update function)

In [ ]:
def top_k_channel_importances_linear_topoplot(point_progression_dict, info, groupby_name="uncertainty", subject_idx=2,save_path="refactor_test/groupby/uncertainty/topoplots/", **kwargs):
    ncols = len(point_progression_dict.keys())
    fig, axs = plt.subplots(nrows=1, ncols=ncols, figsize=(12,4))
    fig.suptitle(f"subject: {subject_idx}")
    for idx, groupby_key in enumerate(point_progression_dict.keys()):
        channel_importancs = list(point_progression_dict[groupby_key][1][-1].values())
        axs[idx].set_title(f"{groupby_name}: {groupby_key}")
        mne.viz.plot_topomap(channel_importancs , info, axes= axs[idx], show=False)

        dir_path = f"{save_path}/linear"
        if not os.path.exists(dir_path):
            os.makedirs(dir_path)

    fig.savefig(f"{dir_path}/top_k_linear_subject_{subject_idx}_topoplot.png")
        


In [ ]:
for i in range(len(subject_indices)):
    top_k_channel_importances_linear_topoplot(points_progression_per_subject_uncertainty[i], info, subject_idx=subject_indices[i], save_path="./groupby/Uncertainty/topoplots")

#### Note here that subjects that have reasonale important channels (subjects 2,7,9 and maybe 11) the importance of the most important channel is more pronounced for the low and medium uncertainty trials.

### same update function

In [ ]:
points_progression_per_subject_uncertainty = []
for i in range(len(subject_indices)):
    points_progression_per_subject_uncertainty.append((top_k_groupby(subjects_dicts[i], "dl_agg", groupby_dicts_list[i][0], ch_names, 10, update_channel_points_same, get_top_k_weighted_individual)))

In [ ]:
def top_k_channel_importances_same_topoplot(point_progression_dict, index, info, groupby_name="uncertainty", subject_idx=2,save_path="refactor_test/groupby/uncertainty/topoplots/", **kwargs):
    ncols = len(point_progression_dict[index].keys())
    fig, axs = plt.subplots(nrows=1, ncols=ncols, figsize=(12,4))
    fig.suptitle(f"subject: {subject_idx}")

    for idx, groupby_key in enumerate(point_progression_dict[index].keys()):
        # subject_idx x groupby_key x all channel values x trial
        # trial set to 0, as all trials contain the same number of channels anyway
       
        channel_importancs = np.array(list(format_point_progression(point_progression_dict[index][groupby_key][1], ch_names).values())).sum(axis=1)
        len(channel_importancs)
        axs[idx].set_title(f"{groupby_name}: {groupby_key}")
        mne.viz.plot_topomap(channel_importancs , info, axes= axs[idx], show=False)

        dir_path = f"{save_path}/same"
        if not os.path.exists(dir_path):
            os.makedirs(dir_path)

    fig.savefig(f"{dir_path}/top_k_same_subject_{subject_idx}_topoplot.png")
        


In [ ]:
for i in range(len(subject_indices)):
    top_k_channel_importances_same_topoplot(points_progression_per_subject_uncertainty, i, info, subject_idx=subject_indices[i], save_path="./groupby/Uncertainty/topoplots")

## General observation.
Using top-k instead of mean to aggreagte, the high uncertainty trials become a lof more noisy.\
That may suggest that the high uncertainty group is dominated by some trials when we aggregate over time_point and trial dimension with mean only.\
However, even after the removal of the trials with extremely high uncertainty, it does not seem like the regions of interest (on the motorcortex) are more important to the model according to the used XAI methods.

# grouby pred fixed

## aggregate over trial and timepoint dimension with mean

In [ ]:
for i in range(len(subject_indices)):
    get_and_plot_topomap_groupby(subjects_dicts[i], groupby_dicts_list[i][1], "dl_agg", np.mean, info, subject_idx=subject_indices[i], save_path="./groupby/Prediction_fixed/topoplots", groupby_name="pred_fixed", axis=(0,2))

## aggregate over timepoint dimension with mean, and over trial dimension with top-k

### linear update function

In [ ]:
points_progression_per_subject_pred_fixed = []
for i in range(len(subject_indices)):
    points_progression_per_subject_pred_fixed.append((top_k_groupby(subjects_dicts[i], "dl_agg", groupby_dicts_list[i][1], ch_names, 10, update_channel_points_linear)))

In [ ]:
for i in range(len(subject_indices)):
    top_k_channel_importances_linear_topoplot(points_progression_per_subject_pred_fixed[i], info, subject_idx=subject_indices[i], groupby_name="pred_fixed",save_path="./groupby/Prediction_fixed/topoplots")

#### Of interest here is that both contralateral sides of (presumably) the most reasonable subjects (2 and 7) one the motor cortex are marked as important here for the "prediction: one" condition compared to the "prediction: zero" condtion

#### Also note that i general, possibly noisy trials can be detected, subject 11 for example shows an obvious eye-blink artifact. Subjects 4, 22 and 24 may also result from noise but I am way less sure in these case.

### same update function

In [ ]:
for i in range(len(subject_indices)):
    top_k_channel_importances_same_topoplot(points_progression_per_subject_pred_fixed,i, info, subject_idx=subject_indices[i], groupby_name="pred_fixed",save_path="./groupby/Prediction_fixed/topoplots")

# grouby pred rolling

## mean

In [ ]:
for i in range(len(subject_indices)):
    get_and_plot_topomap_groupby(subjects_dicts[i], groupby_dicts_list[i][2], "dl_agg", np.mean, info, subject_idx=subject_indices[i], groupby_name="pred_rolling", save_path="./groupby/Prediction_Fixed/topoplots", axis=(0,2))

## linear

In [ ]:
points_progression_per_subject_pred_rolling = []
for i in range(len(subject_indices)):
    points_progression_per_subject_pred_rolling.append((top_k_groupby(subjects_dicts[i], "dl_agg", groupby_dicts_list[i][2], ch_names, 10, update_channel_points_linear)))

In [ ]:
for i in range(len(subject_indices)):
    top_k_channel_importances_linear_topoplot(points_progression_per_subject_pred_rolling[i], info, subject_idx=subject_indices[i], groupby_name="pred_rolling",save_path="./groupby/Prediction_rolling/topoplots")

## same

In [ ]:
for i in range(len(subject_indices)):
    top_k_channel_importances_same_topoplot(points_progression_per_subject_pred_rolling,i, info, subject_idx=subject_indices[i], groupby_name="pred_rolling",save_path="./groupby/Prediction_rolling/topoplots")

# groupby true label

## mean

In [ ]:
for i in range(len(subject_indices)):
    get_and_plot_topomap_groupby(subjects_dicts[i], groupby_dicts_list[i][3], "dl_agg", np.mean, info, subject_idx=subject_indices[i], groupby_name="true", save_path="./groupby/True_label/topoplots", axis=(0,2))

## linear

In [ ]:
points_progression_per_subject_true = []
for i in range(len(subject_indices)):
    points_progression_per_subject_true.append((top_k_groupby(subjects_dicts[i], "dl_agg", groupby_dicts_list[i][3], ch_names, 10, update_channel_points_linear)))

In [ ]:
for i in range(len(subject_indices)):
    top_k_channel_importances_linear_topoplot(points_progression_per_subject_true[i], info, subject_idx=subject_indices[i], groupby_name="true",save_path="./groupby/True_label/topoplots")

## same

In [ ]:
for i in range(len(subject_indices)):
    top_k_channel_importances_same_topoplot(points_progression_per_subject_true,i, info, subject_idx=subject_indices[i], groupby_name="true",save_path="./groupby/True_label/topoplots")

ToDo: Just like for distribution of uncertanties/variancs, check distributions of the raw labels fixed/rolling too. Possibly there are outliers too. If yes, would they have to be removed too